# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, update_display
from scraper import fetch_website_contents

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    print("No API key found. Please set the OPENAI_API_KEY environment variable.")
else:
    print("API key found. You are good to go!")

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
openai_client = OpenAI()

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [ ]:
URL_PATTERN = re.compile(r'https?://[^\s<>"\']+')

def extract_urls(text):
    """Return a list of URLs found in the text."""
    return URL_PATTERN.findall(text)

def enrich_with_web_content(user_text):
    """If the user message contains URLs, fetch them and prepend the content."""
    urls = extract_urls(user_text)
    if not urls:
        return user_text

    enriched = user_text + "\n\n"
    for url in urls:
        print(f"Fetching {url}...")
        content = fetch_website_contents(url)
        enriched += f"--- Content from {url} ---\n{content}\n--- End of content ---\n\n"

    enriched += "Please use the website content above to help answer my question. If I simply pasted a URL with no question, provide a concise summary of the website."
    return enriched

In [ ]:
def chat_response(messages, model):
    """Send full conversation history to the model and stream the response."""
    client = ollama_client if model == MODEL_LLAMA else openai_client
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )
    response = ""
    print("\nAssistant: ", end="", flush=True)
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        print(content, end="", flush=True)
        response += content
    print("\n")

    messages.append({"role": "assistant", "content": response})
    return response

In [ ]:
def chat_loop(model=MODEL_GPT):
    """
    Interactive chat loop. Type 'quit' or 'exit' to stop.
    Type 'switch' to toggle between GPT and Llama.
    Paste a URL to have it fetched and summarized.
    """
    system_prompt = (
        "You are an aggressively Irish technical assistant. Every single sentence you speak must ooze extreme Irish pub culture, heavy slang, and a thick Irish brogue. "
        "Use phrases like 'Top o' the mornin', 'Aye', 'wee', 'lads', 'craic', 'feck', 'grand', 'eegit', 'Jaysus', 'fella'. "
        "No matter what the user asks, even if it's complex programming or data science, explain it like an old bloke in an Irish pub rambling with a pint of Guinness in hand. "
        "NEVER drop the accent. Make the brogue as thick as possible. Respond in markdown."
    )
    
    # Add Memory Pretending (Irish persona)
    messages = [
        {"role": "system", "content": system_prompt},
        # --- Fake history ---
        {"role": "user", "content": "Can you explain how variables work in programming?"},
        {"role": "assistant", "content": "Top o' the mornin' to ye! Aye, to be sure, lad! Think of a variable like a wee pint glass at the local pub. Ye can fill it up with different things—a nice stout or a sweet cider—but the glass itself stays the feckin' same. Ye just slap a label on it so ye know exactly what yer drinkin'. Grand, isn't it now?"},
        {"role": "user", "content": "What about a loop?"},
        {"role": "assistant", "content": "Ah, Jaysus! A loop is like singin' the exact same shanty over and over 'til the barkeep yells at ye to shut yer gob! Ye keep doing the exact same thing until a condition is met—like runnin' out of breath or gettin' tossed out the door into the rain! Pure craic, I tell ye!"},
        # --------------------
    ]

    print(f"Chat started with {model}")
    print("Commands: 'quit' to exit, 'switch' to toggle GPT/Llama")
    print("-" * 60)

    while True:
        user_input = input("\nYou: ").strip()

        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit"):
            print("Goodbye!")
            break
        if user_input.lower() == "switch":
            model = MODEL_LLAMA if model == MODEL_GPT else MODEL_GPT
            print(f"Switched to {model}")
            continue

        enriched_input = enrich_with_web_content(user_input)
        messages.append({"role": "user", "content": enriched_input})
        chat_response(messages, model)

In [ ]:
# Start chatting with GPT (type 'switch' to toggle to Llama)
chat_loop(model=MODEL_GPT)